# 🗣️ Fine-Tuning TTS para Lenguas Low-Resource

**Objetivo:** Crear voces sintéticas para lenguas con pocos recursos digitales
usando fine-tuning de modelos Text-to-Speech.

**Opciones evaluadas:**
1. **MMS-TTS** (Facebook) — 1100+ lenguas, fine-tuning con ~30 min de audio
2. **Coqui XTTS-v2** — Fine-tuning con solo 3-5 min de audio
3. **YourTTS** — Modelo multilingüe, fine-tuning eficiente

**Recomendación:** MMS-TTS (mejor cobertura) + XTTS-v2 (mejor calidad con pocos datos).

In [ ]:
# === INSTALACIÓN ===
# Descomentar según la opción elegida

# Opción 1: MMS-TTS (Facebook)
# !pip install -q transformers accelerate soundfile torch

# Opción 2: Coqui XTTS-v2
# !pip install -q TTS

# Opción 3: F5-TTS
# !pip install -q f5-tts huggingface-hub

# Generales
# !pip install -q datasets librosa huggingface_hub

In [ ]:
import os
import torch
import numpy as np
import soundfile as sf
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

---
## Opción 1: MMS-TTS (Facebook)

Usa el modelo `facebook/mms-tts` que soporta 1100+ lenguas.
Fine-tuning con LoRA para adaptar a una lengua/dialecto específico.

**Ventajas:** Cobertura masiva, fine-tuning eficiente con LoRA
**Desventajas:** Calidad de voz genérica

In [ ]:
from transformers import VitsModel, VitsTokenizer, AutoTokenizer, AutoModel

# === CONFIGURACIÓN MMS-TTS ===
LANG_ISO = "wo"      # Código ISO de la lengua
LANG_NAME = "Wolof"  # Nombre

# Cargar modelo base MMS-TTS
# NOTA: MMS-TTS no requiere fine-tuning explícito para lenguas que ya soporta.
# Usa: https://api-inference.huggingface.co/models/facebook/mms-tts
# con parámetro language="wol"

print("MMS-TTS se usa por API (inferenceClient.ts) o carga directa:")
print(f"  Model: facebook/mms-tts")
print(f"  Lengua: {LANG_NAME} ({LANG_ISO})")
print()
print("Para lenguas NO soportadas en MMS-TTS, fine-tune con LoRA:")
print("  https://huggingface.co/blog/mms-tts-finetune")

---
## Opción 2: Coqui XTTS-v2

Fine-tuning con solo 3-5 minutos de audio.
Mejor calidad pero menos cobertura de lenguas.

**Requisitos:** Audio limpio (sin ruido) + transcripción precisa

In [ ]:
def prepare_xtts_dataset(
    audio_dir: str,
    output_csv: str = "metadata_xtts.csv",
):
    """Prepara dataset para XTTS-v2 fine-tuning.
    
    Estructura esperada:
    audio_dir/
    ├── speaker1/
    │   ├── audio_001.wav
    │   ├── audio_002.wav
    │   └── metadata.csv (path|text|speaker)
    └── speaker2/
    
    metadata.csv formato:
    audio_path|text|speaker_name
    """
    import csv
    from pathlib import Path
    
    rows = []
    audio_dir = Path(audio_dir)
    
    for speaker_dir in audio_dir.iterdir():
        if not speaker_dir.is_dir():
            continue
        meta_csv = speaker_dir / "metadata.csv"
        if not meta_csv.exists():
            print(f"  ⚠️ No metadata.csv en {speaker_dir}")
            continue
        
        with open(meta_csv) as f:
            reader = csv.reader(f, delimiter='|')
            for row in reader:
                if len(row) >= 3:
                    rows.append({
                        "audio_file": str(speaker_dir / row[0]),
                        "text": row[1],
                        "speaker": row[2],
                    })
    
    # Escribir CSV combinado
    with open(output_csv, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["audio_file", "text", "speaker"], delimiter='|')
        writer.writeheader()
        writer.writerows(rows)
    
    print(f"✅ Dataset preparado: {len(rows)} samples → {output_csv}")
    return rows

print("\n📋 Preparación de dataset Coqui XTTS-v2:")
print("   Formato: audio_file|text|speaker_name")
print("   Mínimo: 5-10 minutos de audio limpio por locutor")

In [ ]:
def train_xtts(
    dataset_path: str = "metadata_xtts.csv",
    output_path: str = "./xtts-finetuned",
    num_epochs: int = 100,
    batch_size: int = 16,
):
    """Fine-tuning de XTTS-v2 en dataset propio.
    
    NOTA: TTS library tiene su propio CLI:
    ```bash
    tts --model_name tts_models/multilingual/multi-dataset/xtts_v2 \
        --dataset_audio_metadata <metadata.csv> \
        --output_path <output_dir> \
        --num_epochs 100
    ```
    """
    print("=" * 60)
    print("🏋️ ENTRENAMIENTO XTTS-v2")
    print("=" * 60)
    print(f"\nDataset: {dataset_path}")
    print(f"Output: {output_path}")
    print(f"Épocas: {num_epochs}")
    print(f"Batch: {batch_size}")
    print()
    
    try:
        from TTS.tts.configs.xtts_config import XttsConfig
        from TTS.tts.models.xtts import Xtts
        from TTS.utils.audio import AudioProcessor
        print("✅ XTTS-v2 disponible")
        
        # Comando real:
        cmd = (
            f"tts --model_name tts_models/multilingual/multi-dataset/xtts_v2 "
            f"--dataset_audio_metadata {dataset_path} "
            f"--output_path {output_path} "
            f"--num_epochs {num_epochs} "
            f"--batch_size {batch_size} "
            f"--use_wanb False"
        )
        print(f"\nEjecutar:\n  {cmd}")
        
    except ImportError:
        print("⚠️ TTS no instalado. Ejecuta:")
        print("  pip install TTS")
        print()
        print("Luego ejecuta el comando manualmente desde terminal.")
        print("Ejemplo:")
        print(f"  cd /root/proyecto/full-review")
        cmd = (
            f"tts --model_name tts_models/multilingual/multi-dataset/xtts_v2 "
            f"--dataset_audio_metadata {dataset_path} "
            f"--output_path {output_path} "
            f"--num_epochs {num_epochs}"
        )
        print(f"  {cmd}")

train_xtts()

---
## Opción 3: F5-TTS (Recomendada para baja resource)

Modelo moderno que fine-tunea eficientemente con ~30s a 5min de audio.
Mejor relación calidad/datos.

**Referencia:** https://github.com/SWivid/F5-TTS

In [ ]:
def train_f5tts(
    lang_name: str = "Wolof",
    lang_iso: str = "wo",
    audio_dir: str = "./data/tts_training",
    output_dir: str = "./f5tts-finetuned",
):
    """Entrena F5-TTS en una lengua low-resource.
    
    F5-TTS fine-tunea con muy pocos datos (incluso 1 solo audio).
    """
    print("=" * 60)
    print(f"🏋️ F5-TTS Fine-Tuning: {lang_name} ({lang_iso})")
    print("=" * 60)
    print()
    print("Requisitos:")
    print(f"  - Audio(s) en: {audio_dir}")
    print(f"  - Transcripciones: {audio_dir}/metadata.csv")
    print(f"  - Formato metadata.csv: file_name|transcription")
    print()
    print("Comandos:")
    print("  # Instalar:")
    print("  pip install f5-tts huggingface-hub")
    print()
    print("  # Fine-tuning (desde Python):")
    print("  from f5_tts.api import F5TTS")
    print("  model = F5TTS(model_type='F5-TTS', ckpt_path='SWivid/F5-TTS')")  
    print("  # ... ver docs oficiales")
    print()
    print("  # Inferencia:")
    print("  model.infer('Jàmm ngën, nanga def?', ref_audio='ref.wav')")
    print()
    print("📎 Docs: https://github.com/SWivid/F5-TTS")

train_f5tts()

---
## Inferencia: Pipeline TTS Completo

Una vez entrenado el modelo, este pipeline genera audio desde texto.

In [ ]:
def tts_inference(
    text: str,
    lang: str = "wol",
    model_type: str = "mms",  # "mms", "xtts", "f5"
    output_path: str = "output.wav",
    speaker_wav: str = None,  # Para XTTS/F5 (voz de referencia)
) -> str:
    """Genera audio TTS desde texto.
    
    Args:
        text: Texto a sintetizar
        lang: Código ISO 639-3 (e.g. "wol", "ful")
        model_type: Backend a usar
        output_path: Ruta del audio generado
        speaker_wav: Audio de referencia (solo XTTS/F5)
    
    Returns:
        Ruta del archivo generado
    """
    if model_type == "mms":
        # MMS-TTS via transformers
        from transformers import VitsModel, VitsTokenizer
        
        model_id = f"facebook/mms-tts-{lang}" if lang else "facebook/mms-tts"
        try:
            model = VitsModel.from_pretrained(model_id)
            tokenizer = VitsTokenizer.from_pretrained(model_id)
        except:
            # Fallback: modelo genérico + parámetro language
            model = VitsModel.from_pretrained("facebook/mms-tts")
            tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts")
        
        inputs = tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            output = model(**inputs).waveform
        
        sf.write(output_path, output.squeeze().numpy(), samplerate=16000)
        
    elif model_type == "xtts":
        from TTS.api import TTS
        tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
        tts.tts_to_file(
            text=text,
            speaker_wav=speaker_wav,
            language=lang,
            file_path=output_path,
        )
    
    elif model_type == "f5":
        from f5_tts.api import F5TTS
        f5tts = F5TTS(model_type="F5-TTS", ckpt_path="SWivid/F5-TTS")
        f5tts.infer(text, ref_audio=speaker_wav, file_path=output_path)
    
    print(f"✅ Audio generado: {output_path}")
    return output_path

print("✅ Pipeline de inferencia TTS listo")
print("\nEjemplo:")
print('  tts_inference("Jàmm ngën, nanga def?", lang="wol", model_type="mms")')

---
## Evaluación de Calidad TTS

Métricas objetivas (no requieren humanos).

In [ ]:
def evaluate_tts(
    generated_audio: str,
    reference_audio: str = None,
    text: str = None,
) -> dict:
    """Evalúa calidad del TTS generado."""
    metrics = {}
    print("📊 Evaluación TTS")
    print("=" * 40)
    
    # 1. Duración vs esperada
    audio, sr = sf.read(generated_audio)
    duration = len(audio) / sr
    metrics["duration_s"] = round(duration, 2)
    print(f"  Duración: {duration:.2f}s")
    
    # 2. SNR si hay referencia
    if reference_audio:
        ref, sr_ref = sf.read(reference_audio)
        min_len = min(len(audio), len(ref))
        signal = audio[:min_len]
        noise = ref[:min_len] - signal
        snr = 10 * np.log10(np.var(signal) / np.var(noise + 1e-10))
        metrics["snr_db"] = round(snr, 2)
        print(f"  SNR: {snr:.2f} dB")
    
    # 3. WER (ASR del TTS → comparar con texto original)
    if text:
        # Transcribir el audio generado con Whisper
        import whisper
        asr_model = whisper.load_model("small")
        result = asr_model.transcribe(generated_audio, language="wolof")
        transcribed = result["text"].strip()
        print(f"  Texto original: {text}")
        print(f"  ASR transcrito: {transcribed}")
        
        import evaluate
        wer = evaluate.load("wer")
        wer_score = wer.compute(
            predictions=[transcribed.lower()],
            references=[text.lower()],
        )
        metrics["wer"] = round(wer_score, 4)
        print(f"  WER: {wer_score:.4f} (menor = mejor)")
    
    print("=" * 40)
    return metrics

print("✅ Evaluación TTS lista")
print("\nEjemplo:")
print('  evaluate_tts("output.wav", text="Jàmm ngën, nanga def?")')

---
## Resumen: Estrategia TTS por Lengua

| Lengua | MMS-TTS | XTTS-v2 | F5-TTS | Datos necesarios |
|---|---|---|---|---|
| **Wolof (wo)** | ✅ Nativo | ✅ Fine-tune | ✅ Fine-tune | ~5-30 min audio |
| **Fula (ff)** | ✅ Nativo | 🔄 Pendiente | 🔄 Pendiente | ~5-30 min audio |
| **Bambara (bm)** | ✅ Nativo | 🔄 Pendiente | 🔄 Pendiente | ~5-30 min audio |
| **Serer (srr)** | ❌ No soportado | 🔄 Pendiente | 🔄 Pendiente | ~30-120 min audio |
| **Jola (dyo)** | ❌ No soportado | 🔄 Pendiente | 🔄 Pendiente | ~30-120 min audio |
| **Soninké (snk)** | ❌ No soportado | 🔄 Pendiente | 🔄 Pendiente | ~30-120 min audio |

**Recomendación general:**
1. Para lenguas soportadas por MMS-TTS: usar API directamente (sin fine-tuning)
2. Para lenguas no soportadas: fine-tuning con XTTS-v2 o F5-TTS
3. Para calidad superior en cualquier lengua: XTTS-v2 con ~10min de voz de locutor nativo